# DiD-BCF — D_staggered (linearity_degree=1)

**Workstream D · staggered adoption (cohort x event-time effects)**

dynamic, cohort-varying effects (Goodman-Bacon case)

Fits DiD-BCF on the `D_staggered` scenario at `linearity_degree=1` and reports
metrics for **both** the plain DiD-BCF posterior and the proposed **posterior
correction** (Algorithm 1 of the theory note), so the correction can be judged
directly. Panel: N=200, 4 pre + 4 post periods.

> **Colab:** upload just this notebook and *Run all* — the first cell installs the
> dependencies and the second clones the engine automatically.

In [1]:
# Colab: install the DiD-BCF dependencies (stochtree provides the BCF sampler).
%pip install -q stochtree scikit-learn joblib tqdm pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 16.6 MB/s eta 0:00:00


In [2]:
import os, sys

# --- Locate the DiD-BCF engine ------------------------------------------------
# So you can upload just THIS notebook to Colab and Run all. Resolution order:
#   1. `did_bcf_revision` already importable;
#   2. running inside a repo checkout (the parent folder holds the package);
#   3. otherwise clone https://github.com/hugogobato/DiD-BCF and use it.
REPO_URL = "https://github.com/hugogobato/DiD-BCF.git"
ENGINE_SUBDIR = os.path.join("DiD-BCF", "Simulation_Studies_Revision")

def _locate_root():
    try:
        import did_bcf_revision  # noqa: F401
        return os.path.dirname(os.path.dirname(did_bcf_revision.__file__))
    except Exception:
        pass
    parent = os.path.abspath(os.path.join(os.getcwd(), ".."))
    if os.path.isdir(os.path.join(parent, "did_bcf_revision")):
        return parent
    if not os.path.isdir("DiD-BCF"):
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    return os.path.abspath(ENGINE_SUBDIR)

ROOT = _locate_root()
sys.path.insert(0, ROOT)
print("Using DiD-BCF engine at:", ROOT)

from did_bcf_revision.runner import run_named
from did_bcf_revision.metrics import (compute_metrics, plain_vs_corrected,
                                      surface_metrics)

Using DiD-BCF engine at: /content/DiD-BCF/Simulation_Studies_Revision


In [3]:
REPS = 100      # replications (lower for a quick smoke test)
JOBS = 1        # parallel reps (keep 1 on a single-core/GPU Colab)

bcf_params = dict(num_gfr=50, num_mcmc=500, keep_every=5, num_chains=3)

summaries = run_named(
    "D_staggered",
    linearity_degree=1,
    reps=REPS,
    jobs=JOBS,
    bcf_params=bcf_params,
    prop_method="logit",   # pilot propensity for the posterior correction
    n_splits=2,            # cross-fitting folds for the correction
)
summaries.head()

[D_staggered_lin_1] staggered DGP | N=(200,) | reps=100 | 100 fits | jobs=1


D_staggered: 100%|██████████| 100/100 [5:55:32<00:00, 213.33s/fit]

[D_staggered_lin_1] wrote 3000 rows -> /content/DiD-BCF/Simulation_Studies_Revision/Results/summaries_D_staggered_lin_1.csv


,dgp,setting,linearity_degree,N,rep,estimand_type,estimand_id,g,t,k,...,p_bayes,surf_rmse,surf_mae,surf_n,surf_mape,surf_cover95,surf_len95,surf_cover90,surf_len90,true
0,staggered,D_staggered,1,200,0,GATT,g=4_t=4,4.0,4.0,0.0,...,0.258667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.732024
1,staggered,D_staggered,1,200,0,GATT,g=4_t=5,4.0,5.0,1.0,...,0.144667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.824834
2,staggered,D_staggered,1,200,0,GATT,g=4_t=6,4.0,6.0,2.0,...,0.062667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.917643
3,staggered,D_staggered,1,200,0,GATT,g=4_t=7,4.0,7.0,3.0,...,0.028667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.010453
4,staggered,D_staggered,1,200,0,GATT,g=5_t=5,5.0,5.0,0.0,...,0.041333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.147290


In [4]:
# Decomposed metrics: bias, MC SD/variance, RMSE, MAE, MAPE, coverage 90/95,
# interval length, calibration ratio (avg_post_sd/emp_sd), size/power and their
# Monte-Carlo SEs -- for plain AND corrected DiD-BCF.
metrics = compute_metrics(summaries)
plain_vs_corrected(metrics)

,dgp,setting,linearity_degree,N,estimand_type,estimand_id,role,bias_corrected,bias_plain,cover90_corrected,...,len95_corrected,len95_plain,mae_corrected,mae_plain,reject05_corrected,reject05_plain,rmse_corrected,rmse_plain,sd_ratio_corrected,sd_ratio_plain
0,staggered,D_staggered,1,200,ATT,ATT,power,-1.228054,-4.534834,0.00,...,0.784790,1.645724,1.228054,4.534834,1.0,0.41,1.271731,4.541176,0.555280,2.006605
1,staggered,D_staggered,1,200,ES,k=0,power,-0.434517,-3.801693,0.34,...,0.776193,0.864973,0.448521,3.801693,1.0,0.02,0.516300,3.805640,0.654319,2.273447
2,staggered,D_staggered,1,200,ES,k=1,power,-1.441694,-4.944970,0.00,...,0.854110,1.575614,1.441694,4.944970,1.0,0.49,1.483672,4.952700,0.587517,1.940651
3,staggered,D_staggered,1,200,ES,k=2,power,-1.793733,-4.969233,0.00,...,1.090215,2.375142,1.793733,4.969233,1.0,0.49,1.839874,4.980539,0.615964,1.920396
4,staggered,D_staggered,1,200,ES,k=3,power,-1.756238,-4.592009,0.01,...,1.503741,3.071663,1.756238,4.592009,1.0,0.42,1.819406,4.605381,0.706211,2.244113
5,staggered,D_staggered,1,200,GATT,g=4_t=4,power,0.049543,-2.742329,0.82,...,1.422430,1.571095,0.311819,2.742329,1.0,0.00,0.386595,2.745065,0.899830,28.031128
6,staggered,D_staggered,1,200,GATT,g=4_t=5,power,-0.626875,-3.469633,0.47,...,1.432560,2.024704,0.647396,3.469633,1.0,0.00,0.742312,3.478697,0.836845,2.276310
7,staggered,D_staggered,1,200,GATT,g=4_t=6,power,-1.236821,-4.094258,0.08,...,1.443369,2.571796,1.236821,4.094258,1.0,0.03,1.318971,4.107627,0.741900,2.138087
8,staggered,D_staggered,1,200,GATT,g=4_t=7,power,-1.756238,-4.592009,0.01,...,1.503741,3.071663,1.756238,4.592009,1.0,0.42,1.819406,4.605381,0.706211,2.244113
9,staggered,D_staggered,1,200,GATT,g=5_t=5,power,-0.386302,-3.755710,0.71,...,1.572640,1.334517,0.435198,3.755710,1.0,0.17,0.538488,3.767410,1.004338,1.628487


## CATT-surface metrics (the paper's headline RMSE/MAE/MAPE)

Within-replication RMSE/MAE/MAPE over the *individual* treated observations
(mean +/- SD across runs) plus the *pointwise* CATT coverage -- the evidence
that DiD-BCF recovers the heterogeneous effect that GATT-only methods cannot.

In [5]:
surface_metrics(summaries)

,dgp,setting,linearity_degree,N,method,n_reps,avg_n_treated_obs,surf_rmse_mean,surf_rmse_sd,surf_mae_mean,...,surf_mape_mean,surf_mape_sd,surf_cover90_mean,surf_cover90_sd,surf_cover95_mean,surf_cover95_sd,surf_len90_mean,surf_len90_sd,surf_len95_mean,surf_len95_sd
0,staggered,D_staggered,1,200,corrected,100,431.6,2.318567,0.204368,1.798464,...,0.326502,0.017101,0.228395,0.033738,0.276817,0.038280,1.273831,0.131300,1.540530,0.166210
1,staggered,D_staggered,1,200,plain,100,431.6,4.975813,0.240861,4.534834,...,0.858403,0.041363,0.001732,0.004501,0.006632,0.010081,1.866922,0.144243,2.241372,0.159274


## Goodman-Bacon decomposition (TWFE contamination)

How much of a naive TWFE estimate on this DGP comes from the
"already-treated as control" comparisons that bias it.

In [6]:
from did_bcf_revision.dgps import generate_staggered_did
from did_bcf_revision.goodman_bacon import bacon_summary

df0 = generate_staggered_did(seed=0, linearity_degree=1)
bacon_summary(df0)

{'twfe': 5.063198924628939,
 'bacon_total': 5.0631989246289395,
 'weight_treated_vs_untreated': 0.6334173693819004,
 'weight_earlier_vs_later': 0.2391665942217972,
 'weight_already_treated': 0.1274160363963024,
 'components':                    type  treat_group  control_group    weight        dd  \
 0      Earlier_vs_Later          4.0            5.0  0.060205  2.641511   
 1      Earlier_vs_Later          4.0            6.0  0.106778  3.036811   
 2      Earlier_vs_Later          5.0            6.0  0.072184  4.176324   
 3      Later_vs_Earlier          5.0            4.0  0.045153  3.585888   
 4      Later_vs_Earlier          6.0            4.0  0.053389  4.402213   
 5      Later_vs_Earlier          6.0            5.0  0.028874  4.134568   
 6  Treated_vs_Untreated          4.0            inf  0.231731  4.719428   
 7  Treated_vs_Untreated          5.0            inf  0.234982  6.264973   
 8  Treated_vs_Untreated          6.0            inf  0.166704  7.176302   
 
    contribut